<a href="https://colab.research.google.com/github/gyanellaagreda-bd/AnaliticaEmpresarialIntegrada-26.2/blob/main/LAB_D9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 09 — Analítica de Clientes: Procesos Externos

**Semana 9 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía

**Alumno:** GYANELLA AGREDA

**Dataset real:** Online Retail II — UCI Machine Learning Repository  
**Objetivo:** construir un modelo de churn, caracterizar clientes con RFM, aproximar CLV y seleccionar un umbral de campaña según valor económico.

> El notebook debe terminar en una **decisión comercial cuantificada**, no solo en una métrica de machine learning.

## Trabajo individual y entrega

Esta es una práctica dirigida individual. Ejecute el notebook, verifique los resultados y resuelva los retos con sus propias interpretaciones. Las preguntas 1, 2 y 4 se comentan oralmente en clase; escriba las respuestas 3, 5, 6 y 7. Integre las preguntas 8 y 9 en un único cierre ejecutivo. No se requiere informe separado.

Al terminar, suba el archivo .ipynb ejecutado y resuelto a la carpeta `semana09/` de su repositorio personal de GitHub y entregue el enlace directo al archivo. Declare el uso de IA en el notebook. Online Retail II es el caso dirigido; transfiera el procedimiento al proyecto del curso cuando sus datos permitan definir churn.


# 1. Pregunta de negocio

**¿A qué clientes debe priorizar la empresa en una campaña de retención y qué umbral de probabilidad maximiza el valor económico esperado?**

### Definición operacional
- Unidad: cliente.
- Ventana de observación: 180 días previos al corte.
- Horizonte: 90 días posteriores.
- `churn = 1`: el cliente no compra durante el horizonte posterior.
- Features: RFM y variables transaccionales calculadas solo con información histórica.

In [ ]:
%pip install -q ucimlrepo scikit-learn plotly openpyxl


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_score, recall_score,
    roc_curve, precision_recall_curve
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
RANDOM_STATE = 42
print("Entorno listo.")

Entorno listo.


# 2. Descarga y trazabilidad del dataset real

In [ ]:
# Importación completa, con alternativa desde el Excel oficial.
try:
    retail = fetch_ucirepo(id=502)
    df = retail.data.original.copy()
except Exception as exc:
    print("Importación API no disponible; se intenta el archivo oficial:", type(exc).__name__)
    from io import BytesIO
    from urllib.request import urlopen
    from zipfile import ZipFile
    url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
    with urlopen(url, timeout=120) as response:
        blob = response.read()
    with ZipFile(BytesIO(blob)) as z:
        name = next(name for name in z.namelist() if name.lower().endswith('.xlsx'))
        sheets = pd.read_excel(BytesIO(z.read(name)), sheet_name=None)
    df = pd.concat(sheets.values(), ignore_index=True)
print("Dataset: Online Retail II | UCI ID: 502")
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())
print("Dimensión:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.head())


Importación API no disponible; se intenta el archivo oficial: DatasetNotFoundError
Dataset: Online Retail II | UCI ID: 502
Ejecución: 2026-10-14T22:39:15.460087+00:00
Dimensión: (1067371, 8)
Columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Normalización de nombres de columnas

UCI puede devolver pequeñas variaciones en la escritura de los campos. Esta celda identifica las columnas necesarias sin inventar información.

In [ ]:
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_",""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_","")
        if key in normalized:
            return normalized[key]
    return None

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "CustomerID": find_col(df.columns, ["CustomerID", "Customer ID"]),
    "Country": find_col(df.columns, ["Country"]),
}

print(mapping)

required = ["InvoiceNo","Quantity","InvoiceDate","UnitPrice","CustomerID"]
missing = [k for k in required if mapping[k] is None]
if missing:
    raise ValueError(f"No se identificaron columnas obligatorias: {missing}")

rename_map = {v:k for k,v in mapping.items() if v is not None}
df = df.rename(columns=rename_map)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}


# 3. Control de calidad

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

print("Fecha mínima:", df["InvoiceDate"].min())
print("Fecha máxima:", df["InvoiceDate"].max())
print("Nulos CustomerID:", df["CustomerID"].isna().sum())
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())

cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
print("Cancelaciones:", cancel.sum())

Fecha mínima: 2009-12-01 07:45:00
Fecha máxima: 2011-12-09 12:50:00
Nulos CustomerID: 243007
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Cancelaciones: 19494


In [ ]:
clean = df.loc[
    (~cancel) &
    df["CustomerID"].notna() &
    df["InvoiceDate"].notna() &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

clean["CustomerID"] = clean["CustomerID"].astype(str)
clean["Sales"] = clean["Quantity"] * clean["UnitPrice"]

print("Filas válidas:", len(clean))
print("Clientes válidos:", clean["CustomerID"].nunique())
print("Ventas históricas registradas (£):", round(clean["Sales"].sum(), 2))

Filas válidas: 805549
Clientes válidos: 5878
Ventas históricas registradas (£): 17743429.18


### Pregunta 1
¿Por qué una cancelación o una transacción sin CustomerID no debe tratarse como compra válida para RFM y churn?

**Intercambio oral:** comente en clase; no se requiere transcripción.

**Respuesta (Pregunta 1):**

Una **cancelación** (factura que empieza con "C", cantidad negativa) no es una compra: es una devolución o anulación de una venta previa. Si se contara como compra, inflaría Frequency, distorsionaría Recency (el cliente "compró" cuando en realidad devolvió) y restaría artificialmente a Monetary. En churn, una cancelación no demuestra que el cliente siga activo. Aquí hubo 19,494 cancelaciones.

Una transacción **sin CustomerID** (243,007 registros) no se puede asignar a ningún cliente, y RFM y churn son métricas por cliente. Incluirla mezclaría compras de clientes distintos o anónimos en un solo perfil e impediría construir el target (no sabríamos si "ese cliente" compró en el horizonte). Por eso ambas se excluyen: de 1,067,371 filas quedaron 805,549 válidas, de 5,878 clientes.

# 4. Construcción temporal del target

In [ ]:
# Corte propuesto
cutoff = pd.Timestamp("2011-09-01")
obs_days = 180
horizon_days = 90

# Ajuste defensivo si la cobertura del dataset fuera distinta
max_date = clean["InvoiceDate"].max()
min_date = clean["InvoiceDate"].min()

if cutoff + pd.Timedelta(days=horizon_days) > max_date:
    cutoff = (max_date - pd.Timedelta(days=horizon_days)).normalize()

obs_start = cutoff - pd.Timedelta(days=obs_days)
target_end = cutoff + pd.Timedelta(days=horizon_days)

if obs_start < min_date:
    raise ValueError("La cobertura temporal no permite construir la ventana histórica solicitada.")

print("Inicio observación:", obs_start)
print("Fecha de corte:", cutoff)
print("Fin horizonte:", target_end)

Inicio observación: 2011-03-05 00:00:00
Fecha de corte: 2011-09-01 00:00:00
Fin horizonte: 2011-11-30 00:00:00


In [ ]:
hist = clean[
    (clean["InvoiceDate"] >= obs_start) &
    (clean["InvoiceDate"] < cutoff)
].copy()

future = clean[
    (clean["InvoiceDate"] >= cutoff) &
    (clean["InvoiceDate"] < target_end)
].copy()

print("Transacciones históricas:", len(hist))
print("Transacciones futuras:", len(future))
print("Clientes históricos:", hist["CustomerID"].nunique())
print("Clientes con compra futura:", future["CustomerID"].nunique())

Transacciones históricas: 155362
Transacciones futuras: 151993
Clientes históricos: 2772
Clientes con compra futura: 2839


# 5. Features RFM y variables transaccionales

In [ ]:
# Resumen por factura para ticket promedio
invoice_level = (
    hist.groupby(["CustomerID","InvoiceNo"], as_index=False)
        .agg(
            InvoiceDate=("InvoiceDate","max"),
            invoice_sales=("Sales","sum")
        )
)

rfm = (
    hist.groupby("CustomerID")
        .agg(
            last_purchase=("InvoiceDate","max"),
            Frequency=("InvoiceNo","nunique"),
            Monetary=("Sales","sum"),
            Products=("StockCode","nunique") if "StockCode" in hist.columns else ("InvoiceNo","size"),
            Lines=("InvoiceNo","size")
        )
        .reset_index()
)

rfm["Recency"] = (cutoff - rfm["last_purchase"]).dt.days

ticket = (
    invoice_level.groupby("CustomerID")
                 .agg(avg_ticket=("invoice_sales","mean"))
                 .reset_index()
)

rfm = rfm.merge(ticket, on="CustomerID", how="left")

future_customers = set(future["CustomerID"].unique())
rfm["churn"] = (~rfm["CustomerID"].isin(future_customers)).astype(int)

display(rfm.head())
print("Clientes modelados:", len(rfm))
print("Tasa de churn:", f"{rfm['churn'].mean():.2%}")

,CustomerID,last_purchase,Frequency,Monetary,Products,Lines,Recency,avg_ticket,churn
0,12347.0,2011-08-02 08:48:00,3,1603.68,50,64,29,534.56,0
1,12348.0,2011-04-05 10:47:00,1,367.00,5,5,148,367.00,0
2,12352.0,2011-03-22 16:08:00,2,280.66,8,15,162,140.33,0
3,12353.0,2011-05-19 17:47:00,1,89.00,4,4,104,89.00,1
4,12354.0,2011-04-21 13:11:00,1,1079.40,58,58,132,1079.40,1


Clientes modelados: 2772
Tasa de churn: 39.29%


### Pregunta 2
¿Por qué sería leakage utilizar compras posteriores a la fecha de corte para calcular Recency, Frequency o Monetary?

**Intercambio oral:** comente en clase; no se requiere transcripción.

**Respuesta (Pregunta 2):**

Sería **leakage** (fuga de información) porque las compras posteriores al corte son justamente lo que define el target: `churn = 1` significa que el cliente *no* compra en los 90 días siguientes. Si Recency, Frequency o Monetary usaran esas compras, las features contendrían la respuesta que el modelo debe predecir (un cliente con compras futuras tendría Recency baja y Frequency alta). El modelo mostraría métricas artificialmente buenas en el notebook, pero en producción, el día del corte, esa información no existe y el desempeño real se desplomaría. Por eso las features se calculan solo con la ventana histórica (2011-03-05 a 2011-09-01, `< cutoff`) y el target solo con la ventana futura (2011-09-01 a 2011-11-30, `>= cutoff`).

# 6. Segmentación RFM

In [ ]:
def safe_qcut(series, q=5, reverse=False):
    if series.nunique() <= 1:
        return pd.Series((q + 1) // 2, index=series.index, dtype=int)
    # Valores empatados reciben el mismo score; grupos no necesariamente iguales.
    percentile = series.rank(method="average", pct=True)
    score = np.ceil(percentile * q).clip(1, q).astype(int)
    return q + 1 - score if reverse else score

rfm["R_score"] = safe_qcut(rfm["Recency"], q=5, reverse=True)
rfm["F_score"] = safe_qcut(rfm["Frequency"], q=5, reverse=False)
rfm["M_score"] = safe_qcut(rfm["Monetary"], q=5, reverse=False)
rfm["RFM_score"] = rfm[["R_score","F_score","M_score"]].sum(axis=1)

def segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champions"
    if row["R_score"] <= 2 and row["F_score"] >= 4:
        return "At_Risk_High_Value"
    if row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Hibernating"
    if row["R_score"] >= 4 and row["F_score"] <= 2:
        return "New_or_Promising"
    return "Regular"

rfm["segment"] = rfm.apply(segment, axis=1)

segment_summary = (
    rfm.groupby("segment")
       .agg(
           clientes=("CustomerID","nunique"),
           churn_rate=("churn","mean"),
           recency_med=("Recency","median"),
           monetary_med=("Monetary","median")
       )
       .sort_values("clientes", ascending=False)
)

display(segment_summary)

,clientes,churn_rate,recency_med,monetary_med
segment,,,,
Regular,1061,0.389255,50.0,534.580
Hibernating,815,0.571779,126.0,257.700
Champions,550,0.094545,13.0,1861.725
New_or_Promising,245,0.530612,20.0,301.480
At_Risk_High_Value,101,0.277228,87.0,895.460


In [ ]:
fig = px.scatter(
    rfm,
    x="Recency",
    y="Monetary",
    size="Frequency",
    color="segment",
    hover_data=["CustomerID","churn"],
    title="Mapa RFM de clientes"
)
fig.show()

# 7. Proxy de CLV para priorización económica

In [ ]:
# Proxy didáctico: ticket medio × frecuencia mensual × 12; proxy de ingresos, no CLV completo
months_obs = obs_days / 30.4375
rfm["freq_monthly"] = rfm["Frequency"] / months_obs
rfm["CLV_proxy_12m"] = rfm["avg_ticket"] * rfm["freq_monthly"] * 12

display(
    rfm[["CustomerID","avg_ticket","Frequency","CLV_proxy_12m","segment","churn"]]
      .sort_values("CLV_proxy_12m", ascending=False)
      .head(10)
)

,CustomerID,avg_ticket,Frequency,CLV_proxy_12m,segment,churn
1100,14646.0,3949.664000,30,240435.796000,Champions,0
2683,18102.0,4777.428421,19,184189.771583,Champions,0
2391,17450.0,3830.664706,17,132141.971250,Champions,0
35,12415.0,6811.491111,9,124394.856417,Champions,0
1227,14911.0,727.320494,81,119544.214667,Champions,0
862,14156.0,2100.314815,27,115070.997917,Champions,0
2622,17949.0,1729.379167,24,84220.765417,Champions,0
1315,15098.0,13305.500000,3,80997.231250,At_Risk_High_Value,1
2412,17511.0,2701.329286,14,76740.262792,Champions,0
1731,16029.0,1062.418387,31,66830.543292,Champions,0


> **Advertencia metodológica:** `CLV_proxy_12m` es una aproximación para el ejercicio de decisión. Es una extrapolación de ingresos históricos a 12 meses. No incorpora margen individual, supervivencia, tasa de descuento ni costos futuros. Se conserva CLV_proxy_12m como nombre operativo. Este horizonte económico difiere de los 90 días usados para churn.

### Pregunta 3
¿Por qué dos clientes con la misma probabilidad de churn podrían recibir decisiones distintas si sus valores económicos son diferentes?

**Respuesta:**

**Respuesta (Pregunta 3):**

Porque la probabilidad de churn mide **cuán probable es perder al cliente**, pero la decisión depende del **valor esperado**: *probabilidad × valor en juego − costo de actuar*. Contactar cuesta lo mismo (£5 + incentivo) para todos, pero lo que se recupera es proporcional al `CLV_proxy_12m`.

En los datos: un cliente *Champions* tiene un CLV proxy medio de £9,164, mientras que uno *Hibernating* tiene £783. Con la misma probabilidad de churn, perder al primero cuesta unas 12 veces más. El valor recuperable (CLV × 25 % × 30 %) del primero supera con holgura el costo de contacto, y el del segundo lo cubre con mucho menos margen. Un umbral único de probabilidad ignora esta diferencia: la prioridad debe combinar riesgo y valor económico.

# 8. Modelo de churn con scikit-learn

In [ ]:
# CLV_proxy_12m es proporcional a Monetary: se conserva para la decisión,
# pero no se duplica como predictor del modelo.
features = ["Recency", "Frequency", "Monetary", "Products", "Lines", "avg_ticket"]
model_df = rfm.dropna(subset=features + ["churn"]).copy()
X = model_df[features]
y = model_df["churn"]
idx_train, idx_hold = train_test_split(model_df.index, test_size=0.40,
    random_state=RANDOM_STATE, stratify=y)
idx_val, idx_test = train_test_split(idx_hold, test_size=0.50,
    random_state=RANDOM_STATE, stratify=y.loc[idx_hold])
X_train, y_train = X.loc[idx_train], y.loc[idx_train]
X_val, y_val = X.loc[idx_val], y.loc[idx_val]
X_test, y_test = X.loc[idx_test], y.loc[idx_test]
pipe = Pipeline([("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
pipe.fit(X_train, y_train)
proba_val = pipe.predict_proba(X_val)[:, 1]
proba = pipe.predict_proba(X_test)[:, 1]
print("Clientes entrenamiento / validación / prueba:", len(idx_train), len(idx_val), len(idx_test))
print("ROC-AUC en prueba:", round(roc_auc_score(y_test, proba), 4))
print("Average Precision (AP) en prueba:", round(average_precision_score(y_test, proba), 4))
print("El umbral se selecciona solo en validación. La prueba no se usa para reajustarlo.")


Clientes entrenamiento / validación / prueba: 1663 554 555
ROC-AUC en prueba: 0.7255
Average Precision (AP) en prueba: 0.5926
El umbral se selecciona solo en validación. La prueba no se usa para reajustarlo.


In [ ]:
# Evaluación inicial con umbral 0.50 SOLO como referencia
pred_050 = (proba >= 0.50).astype(int)

print(classification_report(y_test, pred_050, digits=4))
print("Matriz de confusión:")
print(confusion_matrix(y_test, pred_050))

              precision    recall  f1-score   support

           0     0.7181    0.7181    0.7181       337
           1     0.5642    0.5642    0.5642       218

    accuracy                         0.6577       555
   macro avg     0.6412    0.6412    0.6412       555
weighted avg     0.6577    0.6577    0.6577       555

Matriz de confusión:
[[242  95]
 [ 95 123]]


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba)

fig_roc = px.area(
    x=fpr, y=tpr,
    title=f"Curva ROC · AUC={roc_auc_score(y_test, proba):.3f}",
    labels={"x":"False Positive Rate","y":"True Positive Rate"}
)
fig_roc.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line_dash="dash")
fig_roc.show()

fig_pr = px.area(
    x=recall_curve, y=precision_curve,
    title=f"Curva Precision-Recall · AP={average_precision_score(y_test, proba):.3f}",
    labels={"x":"Recall","y":"Precision"}
)
fig_pr.show()

### Pregunta 4
¿Qué métrica utilizaría para comparar modelos si churn fuera minoritario: accuracy, ROC-AUC o AP? Justifique según el costo de perder clientes en riesgo.

**Intercambio oral:** comente en clase; no se requiere transcripción.

**Respuesta (Pregunta 4):**

Usaría **AP (Average Precision)**.

- **Accuracy** engaña con clases desbalanceadas: si solo el 10 % abandona, un modelo que prediga "nadie abandona" tendría 90 % de accuracy y sería inútil para retención.
- **ROC-AUC** mide el ranking global, pero con muchos negativos la tasa de falsos positivos se ve pequeña y la métrica resulta optimista cuando la clase de interés es minoritaria.
- **AP** se centra en la clase positiva (los clientes que se van) y resume el compromiso entre precision y recall: perder a un cliente en riesgo (falso negativo) cuesta ingresos, y contactar a quien no se iba (falso positivo) cuesta dinero.

En este caso el churn es 39.3 % (no tan minoritario): ROC-AUC = 0.7255 y AP = 0.5926, frente a una línea base de 0.393. El modelo ordena mejor que el azar, pero de forma moderada. En todo caso, la métrica final debe cerrarse con el valor económico del umbral.

# 9. Del score al umbral de decisión

In [ ]:
cols = ["CustomerID", "segment", "CLV_proxy_12m", "churn"]
val_result = model_df.loc[idx_val, cols].copy()
val_result["proba_churn"] = proba_val
test_result = model_df.loc[idx_test, cols].copy()
test_result["proba_churn"] = proba
display(val_result.head())


,CustomerID,segment,CLV_proxy_12m,churn,proba_churn
1213,14880.0,Hibernating,474.967042,0,0.557962
1419,15304.0,Regular,534.279583,0,0.277227
1944,16473.0,Regular,199.467083,1,0.552662
1258,14984.0,Hibernating,223.573583,0,0.550544
2394,17455.0,Hibernating,267.241250,1,0.595149


## Supuestos económicos del caso

Estos valores **no provienen de UCI**. Son supuestos explícitos para convertir el score en una decisión:

- costo de contacto: **£5** por cliente contactado;
- incentivo: **£15** por verdadero churn que se supone retenido gracias a la intervención;
- efectividad de retención: **25 %**;
- margen recuperable: **30 % del CLV proxy**.

Se evaluarán múltiples umbrales.

El valor es una simulación retrospectiva bajo supuestos, no una ganancia observada ni un efecto causal probado. La evaluación se refiere únicamente al subconjunto indicado. Una campaña real requiere verificar el efecto incremental y validar en otros periodos.


In [ ]:
CONTACT_COST = 5.0
INCENTIVE = 15.0
RETENTION_RATE = 0.25
RECOVERABLE_MARGIN = 0.30

def evaluate_threshold(data, threshold,
                       retention_rate=RETENTION_RATE,
                       contact_cost=CONTACT_COST,
                       incentive=INCENTIVE,
                       recoverable_margin=RECOVERABLE_MARGIN):

    d = data.copy()
    d["contact"] = (d["proba_churn"] >= threshold).astype(int)

    tp_mask = (d["contact"] == 1) & (d["churn"] == 1)

    contacted = int(d["contact"].sum())
    tp = int(tp_mask.sum())
    fp = int(((d["contact"] == 1) & (d["churn"] == 0)).sum())
    fn = int(((d["contact"] == 0) & (d["churn"] == 1)).sum())

    precision = tp / contacted if contacted else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    contact_total = contacted * contact_cost
    expected_retained = tp * retention_rate
    incentive_total = expected_retained * incentive

    recovered_value = (
        d.loc[tp_mask, "CLV_proxy_12m"].sum()
        * retention_rate
        * recoverable_margin
    )

    net_value = recovered_value - contact_total - incentive_total

    return {
        "threshold": threshold,
        "policy": "No contactar" if threshold > 1 else ("Contactar a todos" if threshold == 0 else f"Umbral {threshold:.2f}"),
        "contacted": contacted,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "recovered_value": recovered_value,
        "contact_cost": contact_total,
        "incentive_cost": incentive_total,
        "net_value": net_value
    }

thresholds = np.r_[0.0, np.round(np.arange(0.10, 0.91, 0.05), 2), 1.01]
# 0.0 contacta a todos; 1.01 representa no contactar a nadie.

economic = pd.DataFrame([
    evaluate_threshold(val_result, t)
    for t in thresholds
])

display(economic)

,threshold,policy,contacted,TP,FP,FN,precision,recall,recovered_value,contact_cost,incentive_cost,net_value
0,0.00,Contactar a todos,554,218,336,0,0.393502,1.000000,23488.781591,2770.0,817.50,19901.281591
1,0.10,Umbral 0.10,491,213,278,5,0.433809,0.977064,21584.199163,2455.0,798.75,18330.449163
2,0.15,Umbral 0.15,467,209,258,9,0.447537,0.958716,20471.384378,2335.0,783.75,17352.634378
3,0.20,Umbral 0.20,434,199,235,19,0.458525,0.912844,18362.875266,2170.0,746.25,15446.625266
4,0.25,Umbral 0.25,415,196,219,22,0.472289,0.899083,12010.999706,2075.0,735.00,9200.999706
5,0.30,Umbral 0.30,387,188,199,30,0.485788,0.862385,10985.510109,1935.0,705.00,8345.510109
6,0.35,Umbral 0.35,352,172,180,46,0.488636,0.788991,9208.669303,1760.0,645.00,6803.669303
7,0.40,Umbral 0.40,297,157,140,61,0.528620,0.720183,8157.087159,1485.0,588.75,6083.337159
8,0.45,Umbral 0.45,260,143,117,75,0.550000,0.655963,6794.325713,1300.0,536.25,4958.075713
9,0.50,Umbral 0.50,207,116,91,102,0.560386,0.532110,5570.818872,1035.0,435.00,4100.818872


In [ ]:
# Desempate: preferir menos contactos, luego mayor umbral.
def choose_policy(table):
    return table.sort_values(["net_value", "contacted", "threshold"],
                             ascending=[False, True, False]).iloc[0]
best = choose_policy(economic)
final_result = evaluate_threshold(test_result, float(best["threshold"]))
print("MEJOR POLÍTICA ENTRE LAS EVALUADAS EN VALIDACIÓN")
print(best.to_string())
print("EVALUACIÓN EN PRUEBA SIN REAJUSTAR EL UMBRAL")
display(pd.DataFrame([final_result]))


MEJOR POLÍTICA ENTRE LAS EVALUADAS EN VALIDACIÓN
threshold                        0.0
policy             Contactar a todos
contacted                        554
TP                               218
FP                               336
FN                                 0
precision                   0.393502
recall                           1.0
recovered_value         23488.781591
contact_cost                  2770.0
incentive_cost                 817.5
net_value               19901.281591
EVALUACIÓN EN PRUEBA SIN REAJUSTAR EL UMBRAL


,threshold,policy,contacted,TP,FP,FN,precision,recall,recovered_value,contact_cost,incentive_cost,net_value
0,0.0,Contactar a todos,555,218,337,0,0.392793,1.0,15381.579972,2775.0,817.5,11789.079972


In [ ]:
fig = px.line(
    economic,
    x="threshold",
    y="net_value",
    markers=True,
    title="Valor neto simulado en validación (1.01 = no contactar)",
    labels={
        "threshold":"Umbral de decisión",
        "net_value":"Valor neto esperado (£)"
    }
)
fig.add_vline(x=float(best["threshold"]), line_dash="dash")
fig.show()

In [ ]:
fig = px.line(
    economic,
    x="threshold",
    y=["precision","recall"],
    markers=True,
    title="Trade-off precision–recall según umbral",
    labels={"threshold":"Umbral","value":"Métrica","variable":"Métrica"}
)
fig.show()

### Pregunta 5
¿La política elegida en validación coincide con el umbral 0.50? Explique la diferencia usando el valor económico.

**Respuesta:**

### Pregunta 6
Al aplicar la política elegida al conjunto de prueba, ¿cuántos clientes se contactarían y cuál es el valor neto esperado bajo los supuestos? No extrapole a toda la cartera.

**Respuesta:**


**Respuesta (Pregunta 5):**

No. La política elegida en validación es **"Contactar a todos"** (umbral 0.00), no 0.50. Genera un valor neto de **£19,901.28** (554 contactados, 218 verdaderos churn capturados, recall 100 %), frente a **£4,100.82** con el umbral 0.50 (207 contactados, 116 verdaderos churn capturados, 102 churners perdidos).

La diferencia está en los costos: contactar a un cliente cuesta £5 más £3.75 de incentivo esperado (15 × 25 %) por cada churner contactado, mientras que retener a un churner recupera un valor proporcional a su CLV proxy. Con 0.50 se pierden 102 churners cuyo valor recuperable es mucho mayor que lo que cuesta contactar a los 336 falsos positivos adicionales (£1,680 en contactos). Pasar de 0.50 a contactar a todos suma £17,918 de valor recuperado a cambio de £1,735 más en contactos y £383 más en incentivos: ganancia neta de £15,800. La curva de valor neto cae de forma continua al subir el umbral, por lo que el 0.50 es solo una convención estadística, no el punto óptimo económico.

**Respuesta (Pregunta 6):**

Al aplicar "Contactar a todos" al conjunto de prueba sin reajustar el umbral, se contactarían **555 clientes** (todos los de prueba). El valor neto esperado es de **£11,789.08**: £15,381.58 de valor recuperado − £2,775.00 de costo de contacto − £817.50 de incentivos, con precision 39.28 % (218 verdaderos churn, 337 falsos positivos) y recall 100 %. Estas cifras valen solo para este subconjunto de prueba bajo los supuestos del caso (contacto £5, incentivo £15, retención 25 %, margen recuperable 30 %); no se extrapolan a toda la cartera ni se interpretan como ganancia observada o efecto causal.

# 10. Sensibilidad económica

In [ ]:
scenarios = []
for retention_rate in [0.10, 0.25, 0.40]:
    temp = pd.DataFrame([evaluate_threshold(val_result, t, retention_rate=retention_rate)
                         for t in thresholds])
    selected = choose_policy(temp)
    row = evaluate_threshold(test_result, float(selected["threshold"]), retention_rate=retention_rate)
    row["retention_rate"] = retention_rate
    row["validation_net_value"] = float(selected["net_value"])
    scenarios.append(row)
scenario_df = pd.DataFrame(scenarios)
display(scenario_df[["retention_rate", "policy", "contacted", "precision", "recall", "net_value"]])


,retention_rate,policy,contacted,precision,recall,net_value
0,0.10,Contactar a todos,555,0.392793,1.0,3050.631989
1,0.25,Contactar a todos,555,0.392793,1.0,11789.079972
2,0.40,Contactar a todos,555,0.392793,1.0,20527.527955


In [ ]:
fig = px.bar(
    scenario_df,
    x="retention_rate",
    y="net_value",
    text="policy",
    title="Valor neto en prueba de cada política seleccionada en validación",
    labels={
        "retention_rate":"Efectividad de retención",
        "net_value":"Valor neto simulado en prueba (£)",
        "threshold":"Umbral óptimo"
    }
)
fig.show()

### Pregunta 7
Al comparar efectividades de 10 %, 25 % y 40 %, ¿cambian la política elegida, los clientes contactados y el valor neto en prueba? Explique qué enseña esto sobre predicción y decisión.

**Respuesta:**


**Respuesta (Pregunta 7):**

Con efectividades de 10 %, 25 % y 40 %, **la política elegida no cambia** (siempre "Contactar a todos") y **los clientes contactados tampoco** (555 en los tres casos). Lo que cambia es el **valor neto en prueba**: £3,050.63 (10 %), £11,789.08 (25 %) y £20,527.53 (40 %), es decir, unos £8,740 más por cada 15 puntos de efectividad.

Esto enseña que la decisión es robusta (con un costo de contacto de £5 frente a un CLV proxy alto, contactar a todos conviene incluso con 10 %), pero el resultado económico es muy sensible a un supuesto que el modelo no conoce: la efectividad real de la campaña. El modelo (las probabilidades) es el mismo en los tres escenarios; el valor lo determinan el supuesto de efectividad y la estructura de costos. Buena predicción no equivale a buena decisión: aquí, con ROC-AUC moderado (0.7255), seleccionar por probabilidad no agrega valor frente a contactar a todos, y hay que validar la efectividad real con un piloto.

# 11. Priorización por segmento RFM

In [ ]:
segment_econ = (
    test_result.groupby("segment")
               .agg(
                   clientes=("CustomerID","nunique"),
                   churn_rate=("churn","mean"),
                   proba_media=("proba_churn","mean"),
                   clv_medio=("CLV_proxy_12m","mean"),
                   clv_total=("CLV_proxy_12m","sum")
               )
               .sort_values("clv_total", ascending=False)
)

display(segment_econ)

,clientes,churn_rate,proba_media,clv_medio,clv_total
segment,,,,,
Champions,113,0.115044,0.095179,9163.556129,1.035482e+06
Regular,218,0.389908,0.369099,1557.564598,3.395491e+05
Hibernating,161,0.577640,0.600383,782.900771,1.260470e+05
At_Risk_High_Value,20,0.100000,0.264459,2161.276577,4.322553e+04
New_or_Promising,43,0.581395,0.472799,756.134557,3.251379e+04


### Preguntas 8 y 9 — Integrar en el cierre ejecutivo
8. ¿Qué segmento priorizaría o descartaría? Sustente con el riesgo y el valor económico observados.
9. Indique una limitación y una condición que haría cambiar la decisión.

No repita las respuestas aquí: inclúyalas en un único cierre.


# 12. Informe ejecutivo

In [ ]:
best_threshold = float(best["threshold"])
best_contacted = int(final_result["contacted"])
best_value = float(final_result["net_value"])

print("Resumen automático para apoyar la redacción")
print("-------------------------------------------")
print("Clientes modelados:", len(model_df))
print("Tasa churn:", f"{model_df['churn'].mean():.2%}")
print("ROC-AUC:", f"{roc_auc_score(y_test, proba):.4f}")
print("AP:", f"{average_precision_score(y_test, proba):.4f}")
print("Política elegida en validación:", best["policy"])
print("Clientes evaluados en prueba:", len(test_result))
print("Clientes a contactar:", best_contacted)
print("Valor neto simulado en prueba (£):", f"{best_value:,.2f}")
print("Precision:", f"{final_result['precision']:.2%}")
print("Recall:", f"{final_result['recall']:.2%}")

Resumen automático para apoyar la redacción
-------------------------------------------
Clientes modelados: 2772
Tasa churn: 39.29%
ROC-AUC: 0.7255
AP: 0.5926
Política elegida en validación: Contactar a todos
Clientes evaluados en prueba: 555
Clientes a contactar: 555
Valor neto simulado en prueba (£): 11,789.08
Precision: 39.28%
Recall: 100.00%


Redacte un cierre de **máximo 8 líneas** con:

1. definición de churn y fecha de corte;
2. número de clientes y tasa de churn;
3. ROC-AUC y AP;
4. política elegida en validación;
5. clientes a contactar en prueba;
6. precision y recall;
7. valor neto simulado en prueba bajo los supuestos;
8. segmento RFM prioritario;
9. una limitación;
10. una condición que haría cambiar la decisión.

### Informe ejecutivo

In [ ]:
# Imprime el texto del cierre con las cifras reales de TU ejecución (copiar a la celda markdown siguiente)
seg_prior = segment_econ["clv_total"].idxmax()
print(f"Churn = no comprar en {horizon_days} días tras el corte {cutoff.date()} (observación de {obs_days} días). "
      f"{len(model_df):,} clientes, churn {model_df['churn'].mean():.1%}. "
      f"ROC-AUC {roc_auc_score(y_test, proba):.3f}, AP {average_precision_score(y_test, proba):.3f}. "
      f"Política elegida en validación: {best['policy']}. "
      f"En prueba: {best_contacted} clientes a contactar (de {len(test_result)}), "
      f"precision {final_result['precision']:.1%}, recall {final_result['recall']:.1%}, "
      f"valor neto simulado £{best_value:,.2f}. Segmento prioritario (mayor CLV proxy total): {seg_prior}.")

Churn = no comprar en 90 días tras el corte 2011-09-01 (observación de 180 días). 2,772 clientes, churn 39.3%. ROC-AUC 0.725, AP 0.593. Política elegida en validación: Contactar a todos. En prueba: 555 clientes a contactar (de 555), precision 39.3%, recall 100.0%, valor neto simulado £11,789.08. Segmento prioritario (mayor CLV proxy total): Champions.


**Cierre ejecutivo (incluye respuestas a las Preguntas 8 y 9):**

Churn = no comprar en los 90 días posteriores al corte (2011-09-01, observación de 180 días); se modelaron 2,772 clientes con 39.3 % de churn. La regresión logística logró ROC-AUC 0.7255 y AP 0.5926 en prueba. La política elegida en validación fue "Contactar a todos": en prueba contacta a 555 clientes, con precision 39.3 % y recall 100 %, y un valor neto simulado de £11,789.08 bajo los supuestos del caso (contacto £5, incentivo £15, retención 25 %, margen 30 %). Priorizaría el segmento **Champions** (113 clientes en prueba): aunque su churn es bajo (11.5 %), concentra el 66 % del CLV proxy total (£1.04 M) con un CLV medio de £9,164, de modo que cada churner evitado recupera mucho más de lo que cuesta contactarlo. Con presupuesto limitado, descartaría primero a *At_Risk_High_Value* (solo 20 clientes en prueba y 10 % de churn observado, muestra demasiado pequeña) y luego a *New_or_Promising* y *Hibernating* (churn alto, 58 %, pero CLV medio de ~£750–780). **Limitación:** el CLV proxy extrapola ingresos históricos de 180 días a 12 meses (sin margen real, supervivencia ni descuento) y el valor es una simulación retrospectiva con un solo corte, no un efecto causal demostrado. **Condición de cambio:** la decisión dejaría de ser rentable si la efectividad real de retención cayera por debajo de ~4.8 % o si el costo por contacto superara ~£26.

# 13. Cierre oral
Comparta la decisión de su informe ejecutivo. No se requiere una segunda entrega escrita.


"Recomiendo **contactar a todos los clientes, priorizando el segmento Champions**, porque el valor neto simulado en prueba es **£11,789.08** bajo los supuestos del caso; esta decisión cambiaría si **la efectividad real de retención fuera menor a ~4.8 % o el costo por contacto superara ~£26, en cuyo caso convendría contactar solo a los clientes de mayor riesgo y valor, o no contactar**."


# 14. Evidencia de entrega

Antes de subir a `semana09/`, confirme:

- [ ] Fuente UCI y ID 502 visibles.
- [ ] Limpieza documentada.
- [ ] Target temporal sin leakage.
- [ ] RFM calculado.
- [ ] CLV proxy explicado.
- [ ] Modelo scikit-learn ejecutado.
- [ ] ROC-AUC y AP.
- [ ] Matriz de confusión.
- [ ] Tabla/gráfico de umbrales.
- [ ] Sensibilidad económica.
- [ ] Decisión cuantificada.
- [ ] Limitaciones.
- [ ] Uso de IA declarado.

**Puntaje referencial:** ______ / 20. Este espacio no se utiliza para calificar el laboratorio dirigido.


# Referencias

- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5CG6D.
- Provost, F., & Fawcett, T. *Data Science for Business*.
- Kelleher, J. D., Mac Namee, B., & D'Arcy, A. *Fundamentals of Machine Learning for Predictive Data Analytics*.